In [20]:
import pandas as pd
import numpy as np
import cv2
import mediapipe as mp
import os
import csv

In [21]:
mp_drawing = mp.solutions.drawing_utils # Drawing helpers
mp_holistic = mp.solutions.holistic # Mediapipe Solutions
mp_face_mesh = mp.solutions.face_mesh   # taking the facial features

In [22]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # COLOR CONVERSION BGR 2 RGB
    image.flags.writeable = False                  # Image is no longer writeable
    results = model.process(image)                 # Make prediction
    image.flags.writeable = True                   # Image is now writeable 
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR COVERSION RGB 2 BGR
    return image, results

In [23]:
def draw_styled_landmarks(image, results):
    # Draw face connections
    mp_drawing.draw_landmarks(image,results.face_landmarks,mp_face_mesh.FACEMESH_CONTOURS, 
                             mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1), 
                             mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
                             ) 
    # Draw pose connections
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
                             ) 
    # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
                             ) 
    # Draw right hand connections  
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                             ) 

In [24]:
class_name="happy"

In [6]:

actions=["hello",
"thank_you",
"sorry",
"please",
"yes",
"no",
"i",
"you",
"need",
"food",
"water",
]
max_samples=200
columns = ['class']

for i in range(33):
    columns += [f'pose_x{i}', f'pose_y{i}', f'pose_z{i}', f'pose_v{i}']

for i in range(468):
    columns += [f'face_x{i}', f'face_y{i}', f'face_z{i}']

for i in range(21):
    columns += [f'lh_x{i}', f'lh_y{i}', f'lh_z{i}']

for i in range(21):
    columns += [f'rh_x{i}', f'rh_y{i}', f'rh_z{i}']


In [7]:
with open('coords.csv', mode='w', newline='') as f:
    csv_writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
    csv_writer.writerow(columns)

In [8]:
import time

In [9]:
cap = cv2.VideoCapture(0)
# Set mediapipe model 
with mp_holistic.Holistic(min_detection_confidence=0.7, min_tracking_confidence=0.7) as holistic:
   
    
    for action in actions:
        sample_count=0
        while cap.isOpened() and sample_count < max_samples:
            ret,frame = cap.read()
            frame=cv2.flip(frame,1)
        

        # Make detections
        # passing frame and modle name i,e (holistic )
            image, results = mediapipe_detection(frame, holistic)
        # print(results) as we able to see it is detecting hands or not 
            # print("LEFT:", results.left_hand_landmarks is not None)
            # print("RIGHT:", results.right_hand_landmarks is not None)
        
        # Draw landmarks
            draw_styled_landmarks(image, results)


            try:
                pose=list(np.array([[res.x,res.y,res.z,res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(132))
                face=list(np.array([[res.x,res.y,res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(1404))
                lh=list(np.array([[res.x,res.y,res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(63))
                rh=list(np.array([[res.x,res.y,res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(63))

                row=pose+face+lh+rh
                row.insert(0, action)
                with open('coords.csv', mode='a', newline='') as f:
                    csv_writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
                    csv_writer.writerow(row) 
                sample_count+=1
                cv2.putText(
                    image,
                    f'{action}: {sample_count}/{max_samples}',
                    (10,40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0,255,0),
                    2
                    )
    






            except Exception as e:
                print(f"error is :-{e}")


        # Show to screen
            cv2.imshow('OpenCV Feed', image)


        # Break gracefully
            if cv2.waitKey(10) & 0xFF == ord('q'):
                break
        
        # 5-second break before next action
        break_start = time.time()

        while time.time() - break_start < 5:
            ret, frame = cap.read()
            frame = cv2.flip(frame, 1)

            remaining = 5 - int(time.time() - break_start)

            cv2.putText(
                frame,
                f"Next Action: {remaining}",
                (120, 200),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.5,
                (0, 255, 255),
                3
                )

            cv2.putText(
                frame,
                "Relax... Get Ready!",
                (70, 260),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 0),
                2
                )

            cv2.imshow("OpenCV Feed", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                cap.release()
                cv2.destroyAllWindows()
                exit()

    
    cap.release()
    cv2.destroyAllWindows()
    # print(results.left_hand_landmarks)

In [25]:
import pandas as pd


In [26]:
df=pd.read_csv("coords.csv")


In [27]:
df.columns

Index(['class', 'pose_x0', 'pose_y0', 'pose_z0', 'pose_v0', 'pose_x1',
       'pose_y1', 'pose_z1', 'pose_v1', 'pose_x2',
       ...
       'rh_z17', 'rh_x18', 'rh_y18', 'rh_z18', 'rh_x19', 'rh_y19', 'rh_z19',
       'rh_x20', 'rh_y20', 'rh_z20'],
      dtype='str', length=1663)

In [28]:
df["class"].value_counts()

class
hello        200
thank_you    200
sorry        200
please       200
yes          200
no           200
i            200
you          200
need         200
food         200
water        200
Name: count, dtype: int64

In [29]:
df.shape

(2200, 1663)

In [30]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline 
from sklearn.preprocessing import StandardScaler 
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import pickle
from sklearn.metrics import classification_report, accuracy_score

In [31]:
X = df.drop('class', axis=1) # features
y = df['class'] # target value



In [32]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1234)

In [33]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)


In [34]:
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [35]:
y_pred = model.predict(X_test)

In [36]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9954545454545455
              precision    recall  f1-score   support

        food       0.98      1.00      0.99        60
       hello       1.00      1.00      1.00        59
           i       1.00      1.00      1.00        44
        need       1.00      1.00      1.00        64
          no       0.98      1.00      0.99        59
      please       1.00      1.00      1.00        60
       sorry       1.00      1.00      1.00        61
   thank_you       0.99      1.00      0.99        72
       water       1.00      0.98      0.99        62
         yes       1.00      1.00      1.00        60
         you       1.00      0.97      0.98        59

    accuracy                           1.00       660
   macro avg       1.00      1.00      1.00       660
weighted avg       1.00      1.00      1.00       660



In [23]:
with open('body_language.pkl', 'wb') as f:
    pickle.dump(model, f)

In [37]:
with open('body_language.pkl', 'rb') as f:
    model = pickle.load(f)

In [38]:
from llm import output
from tts_service import speak
predected_senteces=[]
predictions = []
sentence = []
threshold = 0.8
cap = cv2.VideoCapture(0)
# Initiate holistic model
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    
    while cap.isOpened():
        ret, frame = cap.read()
        frame=cv2.flip(frame,1)
        

        # Make detections
        # passing frame and modle name i,e (holistic )
        image, results = mediapipe_detection(frame, holistic)
        # print(results) as we able to see it is detecting hands or not 
            # print("LEFT:", results.left_hand_landmarks is not None)
            # print("RIGHT:", results.right_hand_landmarks is not None)
        
        # Draw landmarks
        draw_styled_landmarks(image, results)
        
        try:
            pose=list(np.array([[res.x,res.y,res.z,res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(132))
            face=list(np.array([[res.x,res.y,res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(1404))
            lh=list(np.array([[res.x,res.y,res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(63))
            rh=list(np.array([[res.x,res.y,res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(63))

            row=pose+face+lh+rh
            
#             # Append class name 
#             row.insert(0, class_name)
            
#             # Export to CSV
#             with open('coords.csv', mode='a', newline='') as f:
#                 csv_writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
#                 csv_writer.writerow(row) 

            # Make Detections
            X = pd.DataFrame([row])
            body_language_class = model.predict(X)[0]
            body_language_prob = model.predict_proba(X)[0]
            confidence = np.max(body_language_prob)
            predictions.append(body_language_class)

            # Last 10 predictions same hone chahiye
            if len(predictions) >= 10:
                if predictions[-10:].count(body_language_class) == 10 and confidence > threshold:
                    if len(sentence) == 0:
                        sentence.append(body_language_class)
                    elif sentence[-1] != body_language_class:
                        sentence.append(body_language_class)
            
            # print(body_language_class, body_language_prob)
            h, w, _ = image.shape
            
            # Grab ear coords

            coords = tuple(np.multiply(
                            np.array(
                                (results.pose_landmarks.landmark[mp_holistic.PoseLandmark.LEFT_EAR].x, 
                                 results.pose_landmarks.landmark[mp_holistic.PoseLandmark.LEFT_EAR].y))
                        , [640,480]).astype(int))
            cv2.rectangle(image, (21,43), (640,40), (0,0,0), -1)
            cv2.putText(image,
                        " ".join(sentence),
                        (10, h - 20),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        1,
                        (255,255,255),
                        2)
            
            cv2.rectangle(image, 
                          (coords[0], coords[1]+5), 
                          (coords[0]+len(body_language_class)*20, coords[1]-30), 
                          (245, 117, 16), -1)
            cv2.putText(image, body_language_class, coords, 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
            
            # Get status box
            cv2.rectangle(image, (0,0), (250, 60), (245, 117, 16), -1)
            
            # Display Class
            cv2.putText(image, 'CLASS'
                        , (95,12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
            cv2.putText(image, body_language_class.split(' ')[0]
                        , (90,40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
            
            # Display Probability
            cv2.putText(image, 'PROB'
                        , (15,12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
            cv2.putText(image, str(round(body_language_prob[np.argmax(body_language_prob)],2))
                        , (10,40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
            
        except:
            pass
                        
        cv2.imshow('Raw Webcam Feed', image)
        if cv2.waitKey(10) & 0xFF == ord(' '):
            final_sentence = output(sentence)
            predected_senteces.append(final_sentence)
            print(final_sentence)
            speak(final_sentence)
        

            sentence = []
            predictions = []

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

c:\Users\user\Desktop\computervision\ai_action_predictor\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\user\Desktop\computervision\ai_action_predictor\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\user\Desktop\computervision\ai_action_predictor\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\user\Desktop\computervision\ai_action_predictor\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\user\Desktop\computervision\ai_action_p

Please.


c:\Users\user\Desktop\computervision\ai_action_predictor\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\user\Desktop\computervision\ai_action_predictor\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\user\Desktop\computervision\ai_action_predictor\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\user\Desktop\computervision\ai_action_predictor\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\user\Desktop\computervision\ai_action_p

In [ ]:
print(predictions)
print(sentence)

['hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'hello', 'thanks', 'thanks', 'thanks', 'thanks', 'thanks', 'thanks', 'thanks', 'thanks', 'hello', 'hello', '

In [ ]:
from llm import output

output(sentence)

'Hello, thanks, and yes.'

In [16]:
from tts_service import speak


In [ ]:
from tts_service import speak
speak("who are you")

None


In [39]:
predected_senteces

['Please.']